In [5]:
import sys
sys.path.append("..")

In [6]:
import torch

## Causal Masking

In [7]:
# 2 batches, 5 tokens each
input_ids = torch.tensor([[1, 0, 0, 2, 3, 4, 5, 0], [6, 7, 8, 9, 10, 0, 0, 0]])

input_ids.shape  # (B, L)

torch.Size([2, 8])

In [8]:
print(input_ids)

tensor([[ 1,  0,  0,  2,  3,  4,  5,  0],
        [ 6,  7,  8,  9, 10,  0,  0,  0]])


In [9]:
from src.model import generate_causal_mask

mask = generate_causal_mask(input_ids.shape[1], input_ids.device)

mask.shape  # (1,1,L,L)

torch.Size([1, 1, 8, 8])

In [10]:
num_heads = 4

B, L = input_ids.shape
H = num_heads  # number of attention heads in your MHA

In [11]:
mask = mask.expand(B, H, L, L)

mask.shape

torch.Size([2, 4, 8, 8])

In [12]:
pad_id = 0

padding_mask = (input_ids != pad_id).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, L)

In [13]:
# True = valid token, False = pad
mask = mask & padding_mask  # (B, H, L, L)

mask.shape

torch.Size([2, 4, 8, 8])

In [14]:
mask

tensor([[[[ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False,  True, False, False, False, False],
          [ True, False, False,  True,  True, False, False, False],
          [ True, False, False,  True,  True,  True, False, False],
          [ True, False, False,  True,  True,  True,  True, False],
          [ True, False, False,  True,  True,  True,  True, False]],

         [[ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False,  True, False, False, False, False],
          [ True, False, False,  True,  True, False, False, False],
          [ True, False, False,  True,  True,  True, False, False],
          [ True, False, False,  True,  True, 

In [28]:
from src.model import get_attn_mask

example = torch.tensor([[1, 1, 1, 1, 1]])

make_attn_mask = get_attn_mask(example, pad_id)

print(make_attn_mask)  # (B, H, L, L)

tensor([[[[ True, False, False, False, False],
          [ True,  True, False, False, False],
          [ True,  True,  True, False, False],
          [ True,  True,  True,  True, False],
          [ True,  True,  True,  True,  True]]]])


In [15]:
B, L, V = 1, 4, 5  # batch size, sequence length, vocabulary size

logits = torch.randn(B, L, V)  # (B, L, V)
labels = torch.tensor([[1, 2, 3, 4]])  # (B, L)

## Logits Shifting for Causal Training

In [16]:
shift_logits = logits[:, :-1, :].contiguous()  # (B, L-1, V)
shift_labels = labels[:, 1:].contiguous()  # (B, L-1)

In [17]:
print(logits)

tensor([[[ 0.8509, -0.7481, -1.1878, -1.5101, -0.3639],
         [ 1.4298, -0.0363, -1.1636, -0.6336,  1.2167],
         [ 0.4827, -0.4241,  0.7262, -0.8933,  0.3587],
         [-2.2790,  0.4710, -0.5943, -1.4112,  0.1357]]])


In [18]:
print(shift_logits)

tensor([[[ 0.8509, -0.7481, -1.1878, -1.5101, -0.3639],
         [ 1.4298, -0.0363, -1.1636, -0.6336,  1.2167],
         [ 0.4827, -0.4241,  0.7262, -0.8933,  0.3587]]])


In [19]:
print(shift_labels)

tensor([[2, 3, 4]])


In [ ]:
shift_logits[0, 0, :], shift_labels[0, 0] # 1st logits predicts the 2nd label

(tensor([ 0.8509, -0.7481, -1.1878, -1.5101, -0.3639]), tensor(2))